In [27]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, HTML

BASE_DIR = Path(".").resolve()
MAPPINGS_DIR = BASE_DIR / "acc_mappings" / "acc_mappings_obx"


In [28]:

rows = []

for fp in sorted(MAPPINGS_DIR.glob("*.json")):
    firm_id = fp.stem
    data = json.loads(fp.read_text(encoding="utf-8"))

    for v in data.get("variables", []):
        if v.get("needs_manual_review", False):
            final_choice = v.get("final_choice", [])
            # make final_choice readable
            final_str = "; ".join([f"{x['sheet_name']}::{x['row_label']}" for x in final_choice]) if final_choice else ""
            rows.append({
                "firm_id": firm_id,
                "variable": v.get("variable", ""),
                "notes": v.get("notes", ""),
                "final_choice": final_str,
                "num_candidates": len(v.get("candidates", [])),
            })

if len(rows) == 0:
    print("No manual reviews needed. All mappings have been reviewed and finalized.")
else:
    df_review = pd.DataFrame(rows).sort_values(["firm_id", "variable"]).reset_index(drop=True)

    print(f"Manual review needed for {len(df_review)} firm-variable mappings across {df_review['firm_id'].nunique() if len(df_review) else 0} firms.")
    display(HTML(f"""
    <div style="max-height: 450px; overflow-y: auto; border: 1px solid #ddd; padding: 6px;">
    {df_review.to_html(index=False)}
    </div>
    """))

Manual review needed for 7 firm-variable mappings across 7 firms.


firm_id,variable,notes,final_choice,num_candidates
PRSO.OL,PPEGT,"No clean total PPE/net PPE line is reported. The balance sheet shows components such as 'Vessels' and 'Equipment - Net', but I cannot identify a confident total PPE label without risking an incomplete mapping.",,5
SNI.OL,PPEGT,"No explicit 'Property, plant and equipment' total is shown. 'Transport-NBV' is the best apparent net PPE-style label, with 'Fleet (Aircraft, Ships etc) gross' as fallback, but coverage of total PPE is uncertain.","Balance Sheet::Transport-NBV; Balance Sheet::Fleet (Aircraft, Ships etc) gross",3
STRO.OL,STD,"Selected the cleanest reported current debt line. Manual review recommended because 'Bank overdraft (credit facilities)' may be an additional short-term debt component in some years, and inclusion overlap is unclear from the preview.",Balance Sheet::Current interest-bearing liabilities,4
SUBC.OL,PPEGT,"No clean single total PPE line is reported. Chose the tangible fixed-asset component block from the balance sheet as the best approximation, but presentation is ambiguous because some items are gross while 'Vessels' may already be net; manual review is advisable.","Balance Sheet::Vessels; Balance Sheet::Operating Equip, Gross; Balance Sheet::Land/Buildings, Gross; Balance Sheet::Other; Balance Sheet::Depreciation",6
SWON.OL,STD,"No clean 'Borrowings, current' label is present. 'Other current finacial liabilities' appears to be the best current financing bucket and leases are separately disclosed, so adding it with 'Bank overdrafts' is plausible, but the label is broad enough to warrant review for possible non-debt content or overlap.",Balance Sheet::Other current finacial liabilities; Balance Sheet::Bank overdrafts,2
VISTN.OL,PPEGT,"No clean reported total/net PPE line is available; balancing-value rows look unusable. Selected the likely non-lease PPE components as a provisional sum, but classification/double-counting risk remains.","Balance Sheet::Plant, Machinery & Equipment - Gross; Balance Sheet::Construction in Progress - Gross; Balance Sheet::Property, Plant & Equipment - Gross",6
VOW.OL,PPEGT,Chose 'Fixed assets' outside the shortlist because it is the only balance-sheet tangible fixed asset/PPE stock line and is preferable to lease-specific or cash-flow depreciation lines. | Auto-flag: best candidate confidence 0.20 < 0.65,Balance Sheet::Fixed assets,5
